In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
)

**Setup and model load**

In [2]:
MODEL_DIR = OUTPUT_DIR = "/content/drive/MyDrive/Fine tuned models/vit_beans_best"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_DIR
).to(device)

model.eval()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [3]:
dataset = load_dataset("AI-Lab-Makerere/beans")

print("Device:", device)
print("Classes:", model.config.id2label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Device: cuda
Classes: {0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'}


**Prediction function**

In [4]:
def predict(image):
    inputs = processor(
        images=image.convert("RGB"),
        return_tensors="pt"
    )
    pixel_values = inputs["pixel_values"].to(device)
    with torch.inference_mode():
        logits = model(pixel_values=pixel_values).logits
    
    probabilities = torch.softmax(logits, dim=-1)[0]
    predicted_id = probabilities.argmax().item()

    return probabilities.cpu(), predicted_id